# Variables at a fixed period

Some relations refer to a variable in **one particular period**, the same period whatever the
period being calculated: an index with 2015 = 100, a value in the prices of a base year, a
share calibrated to the level of a reference year. ModelFlow writes this as `X(2015)`, the value
of `X` in 2015:

```text
> index = x/x(2015)*100
```

Before, the usual workaround was an extra exogenous variable holding the 2015 value in every
row. `X(2015)` does the same without the extra column, and it also works when `X` is endogenous.

This chapter covers the syntax, what happens in a model, which solvers can be used, and how
fixed periods are handled in estimation.

In [1]:
import numpy as np
import pandas as pd

from modelclass import model
from modelconstruct_estimation import Makemodel
from modelestimator_new import Estimate_ols, Estimate_nls_lmfit

## The syntax

A fixed period is written in the parentheses where a lag or a lead goes. The difference is the
sign: a lag or a lead **always** has a sign, a fixed period never does.

| Written | Meaning |
|---|---|
| `X(-1)` | `X` one period before the current period (a lag) |
| `X(+1)` | `X` one period after the current period (a lead) |
| `X(2015)` | `X` in 2015: an unsigned number with at least 4 digits is a year |
| `X('2015Q1')` | `X` in the period with the index label `'2015Q1'` -- a quarterly model |
| `X('2015-03')` | `X` in March 2015 -- a monthly model |

A period in quotes is the key used to index the DataFrame, exactly as in `df.loc['2015Q1']`.
`X('2015')` and `X(2015)` are the same period. A short label, like `X('5')`, needs the quotes.

Both forms are legal Python, so equations with fixed periods can also be read by the estimators,
see the last section.

## In a model

A small recursive model: `Y` is `X` indexed to 2002, and `Z` adds the 2005 value of `X` to
last period's `Y`.

In [2]:
years = list(range(2000, 2011))
df = pd.DataFrame({'X': np.linspace(10, 20, len(years))}, index=years)

mfix = model("""
Y = X/X(2002)*100
Z = Y(-1) + X(2005)
""", modelname='fixdemo')

res = mfix(df, 2003, 2010)
res

Will start solving: fixdemo
makelos_ng compiles the xgenr (DAG) evaluator for fixdemo
2003  solved
2004  solved
2005  solved
2006  solved
2007  solved
2008  solved
2009  solved
2010  solved
fixdemo solved  


,X,Y,Z
2000,10.0,0.000000,0.000000
2001,11.0,0.000000,0.000000
2002,12.0,0.000000,0.000000
2003,13.0,108.333333,15.000000
2004,14.0,116.666667,123.333333
2005,15.0,125.000000,131.666667
2006,16.0,133.333333,140.000000
2007,17.0,141.666667,148.333333
2008,18.0,150.000000,156.666667
2009,19.0,158.333333,165.000000


`Y` is 100 where `X` equals its 2002 value, and `Z` carries the 2005 value of `X` in every
period. The model knows which variables it reads at a fixed period, and which periods they are:

In [3]:
print(f'{mfix.fixed_terms=}')
print(f'{mfix.fixed_periods=}')
print(f'solved with {mfix.model_solver.__name__}')

mfix.fixed_terms=[('X', '2002'), ('X', '2005')]
mfix.fixed_periods=['2002', '2005']
solved with xgenr_ng


### How it works

Only the **ng solvers** (`xgenr_ng`, `sim_ng`, `newton_ng`, `newtonstack_ng`, ...) can evaluate a
fixed period. When a model refers to one, calling the model picks the ng version of the solver it
would otherwise have used. The legacy solvers raise an exception.

The generated code reads `X(2002)` from the **row of 2002** in the data -- a constant, while a lag
is read relative to the row being calculated. Here `values[2,...]` is `X` in 2002, the third row:

In [4]:
print(mfix.make_los_text)

def make_los(funks=[],errorfunk=None):
    from modeluserfunk import recode
    from modelBLfunk import array, cdf_lognorm_econ, clognorm, erf, exp, gamma, isclose, lifetime_credit_loss, log, logit, logit_inverse, lognorm, matrix, mv_opt, mv_opt_prop, norm, normcdf, part_exp_lognorm, qgamma, sqrt, sum_excel, tanh, transpose
    def los(values,row,solveorder, allvar):
        try :
            values[row,1]=values[row,0]/values[2,0]*100
            values[row,2]=values[row-1,1]+values[5,0]
        except :
            errorfunk(values,sys.exc_info()[2].tb_lineno,overhead=5,overeq=0)
            raise
        return 
    return los



Because the row number is compiled into the code, the code is generated again when the same
columns come with another index -- here the DataFrame starts in 2001 instead of 2000, and the
result is the same. The period must be in the DataFrame; if it is not, the solve stops with an
error:

In [5]:
late = mfix(df.loc[2001:], 2003, 2010)
print('same result with another index:', np.allclose(late.loc[2003:, 'Y'], res.loc[2003:, 'Y']))

try:
    mfix(df.loc[2003:], 2004, 2010)
except Exception as e:
    print(f'{type(e).__name__}: {e}')

Will start solving: fixdemo
makelos_ng compiles the xgenr (DAG) evaluator for fixdemo
2003  solved
2004  solved
2005  solved
2006  solved
2007  solved
2008  solved
2009  solved
2010  solved
fixdemo solved  
same result with another index: True
Will start solving: fixdemo
KeyError: 'The period 2002 in a reference like X(2002) is not a row in the dataframe index (2003 to 2010)'


### On a DataFrame with `mfcalc`

`mfcalc` runs equations directly on a DataFrame, so a fixed period works there too:

In [6]:
df.mfcalc('IDX = X/X(2002)*100')

,X,IDX
2000,10.0,83.333333
2001,11.0,91.666667
2002,12.0,100.000000
2003,13.0,108.333333
2004,14.0,116.666667
2005,15.0,125.000000
2006,16.0,133.333333
2007,17.0,141.666667
2008,18.0,150.000000
2009,19.0,158.333333


### Quarterly and monthly data

With a quarterly or monthly index the period is given as text, the label the DataFrame is
indexed with:

In [7]:
qidx = pd.period_range('2001Q1', '2004Q4', freq='Q')
dfq = pd.DataFrame({'X': np.linspace(10, 25, len(qidx))}, index=qidx)

mq = model("Y = X/X('2002Q1')*100", modelname='fixq')
mq(dfq, '2002Q2', '2004Q4').loc['2002Q1':'2003Q2']

Will start solving: fixq
makelos_ng compiles the xgenr (DAG) evaluator for fixq
2002Q2  solved
2002Q3  solved
2002Q4  solved
2003Q1  solved
2003Q2  solved
2003Q3  solved
2003Q4  solved
2004Q1  solved
2004Q2  solved
2004Q3  solved
2004Q4  solved
fixq solved  


,X,Y
2002Q1,14.0,0.000000
2002Q2,15.0,107.142857
2002Q3,16.0,114.285714
2002Q4,17.0,121.428571
2003Q1,18.0,128.571429
2003Q2,19.0,135.714286


## An endogenous variable at a fixed period inside the solve span

So far the fixed periods were data: `X` is exogenous, or the period lies before the periods
solved. It is different when the variable is **endogenous** and the period is **inside** the
solve span. In

```text
> y = 0.5*y(-1) + x
> z = y/y(2006)*100
```

solved from 2003 to 2010, `z` in 2003 needs `y` in 2006 -- a value which is only solved three
periods later.

A solver which solves **one period at a time** (`xgenr_ng`, `sim_ng`, `newton_ng`,
`newton_fbmin_ng`) can not do that: in the periods before 2006 it would read the starting value
of `y` in 2006. These solvers stop with an exception.

The **stacked** solvers, `newtonstack_ng` and `newtonstack_fbmin_ng`, solve **all periods at
once**. For them `y(2006)` is just one more unknown -- a column of the stacked Jacobian -- and the
model is solved exactly. Calling the model picks a one-period solver, so the stacked solver must be
asked for with `solver=`:

In [8]:
dfi = pd.DataFrame({'X': np.linspace(1, 3, len(years)), 'Y': 1.0, 'Z': 0.0}, index=years)
mins = model("""
Y = 0.5*Y(-1) + X
Z = Y/Y(2006)*100
""", modelname='fixinspan')

try:
    mins(dfi, 2003, 2010)
except Exception as e:
    print(f'{type(e).__name__}: {e}')

Will start solving: fixinspan
Exception: Y(2006) refers to an endogenous variable at a fixed period inside the solve span 2003 to 2010. A per-period solver can not solve that.
Use a stacked solver (newtonstack_ng or newtonstack_fbmin_ng), solve a span without the period, or let an exogenous variable hold the value.


In [9]:
res_inspan = mins(dfi, 2003, 2010, solver='newtonstack_ng')
res_inspan.loc[2003:2010, ['X', 'Y', 'Z']]

,X,Y,Z
2003,1.6,2.100000,53.674121
2004,1.8,2.850000,72.843450
2005,2.0,3.425000,87.539936
2006,2.2,3.912500,100.000000
2007,2.4,4.356250,111.341853
2008,2.6,4.778125,122.124601
2009,2.8,5.189063,132.627796
2010,3.0,5.594531,142.991214


`Z` is 100 in 2006, as it must be, and in the other periods it is `Y` relative to the **solved**
value of `Y` in 2006.

The same holds when the fixed period feeds back into the simultaneous part of a model. Here
consumption depends on income in 2006, which itself depends on consumption -- both stacked solvers
find the same solution:

In [10]:
msim = model("""
C = 0.6*Y + 0.2*C(-1) + 0.1*Y(2006)
Y = C + G
Z = Y/Y(2006)*100
""", modelname='fixinspan_sim')
dfs = pd.DataFrame({'G': np.linspace(20, 30, len(years)), 'C': 60.0, 'Y': 100.0, 'Z': 0.0},
                   index=years)

stack = msim.newtonstack_ng(dfs, 2003, 2010)
fbmin = msim.newtonstack_fbmin_ng(dfs, 2003, 2010)
print('largest difference between the two:',
      (stack - fbmin).loc[2003:2010, ['C', 'Y', 'Z']].abs().max().max())
stack.loc[2003:2010, ['G', 'C', 'Y', 'Z']]

The feedback variables are not directly connected, so the feedback order is left as found
largest difference between the two: 8.526512829121202e-14


,G,C,Y,Z
2003,23.0,111.941176,134.941176,71.109733
2004,24.0,139.411765,163.411765,86.112833
2005,25.0,154.647059,179.647059,94.668320
2006,26.0,163.764706,189.764706,100.000000
2007,27.0,169.823529,196.823529,103.719777
2008,28.0,174.352941,202.352941,106.633602
2009,29.0,178.117647,207.117647,109.144451
2010,30.0,181.500000,211.500000,111.453813


Which solvers accept what:

| | Exogenous `X(P)`, or `P` outside the solve span | Endogenous `X(P)`, `P` inside the solve span |
|---|---|---|
| `xgenr_ng`, `sim_ng`, `newton_ng`, `newton_fbmin_ng` | yes | no -- an exception |
| `newtonstack_ng`, `newtonstack_fbmin_ng` | yes | yes, exact |
| `res_ng` (evaluates the equations, does not solve) | yes | yes -- reads the input values |
| legacy solvers (`sim`, `newton`, ...) and `sim1d_ng` | no | no |

A span which starts **after** the period is always fine: then `Y(2006)` is data.

## Rules, and what is caught when a model is made

* A fixed period can not be on the left hand side: `Y(2002) = X` makes no sense as an equation.
* No blanks inside the parentheses: `X( 2002 )` and `X( -1 )` are rejected.
* An unsigned number of 1 to 3 digits, `X(1)`, is not allowed in a model. It is the EViews way to
  write a lead; when equations are normalized (`fixleads`), such leads get their sign, `X(+1)`.
  From 4 digits on, an unsigned number is a year.
* A sub-annual period must be in quotes: `X(2002Q1)` is an error, `X('2002Q1')` is right.

All of these are caught when the model is made, not when it is solved:

In [11]:
for text in ['Y(2002) = X', 'Y = X/X( 2002 )*100', 'Y = X(1) + Z', 'Y = X/X(2002Q1)']:
    try:
        model(text)
        print(f'{text:22} accepted')
    except Exception as e:
        print(f'{text:22} -> {str(e).splitlines()[0]}')

Y(2002) = X            -> A variable at a fixed period (like X(2002)) can not be on the left hand side:
Y = X/X( 2002 )*100    -> X( : X is not a function. Lags and leads are written with a sign, X(-1) or X(+1), a fixed period as a year, X(2002), or as text, X('2002Q1'), all without blanks:
Y = X(1) + Z           -> X( : X is not a function. Lags and leads are written with a sign, X(-1) or X(+1), a fixed period as a year, X(2002), or as text, X('2002Q1'), all without blanks:
Y = X/X(2002Q1)        -> X( : X is not a function. Lags and leads are written with a sign, X(-1) or X(+1), a fixed period as a year, X(2002), or as text, X('2002Q1'), all without blanks:


## In estimation

The estimators read an equation with Python's own parser. `X(2005)` and `X('2005Q1')` are legal
Python -- a call of `X`, just like the lag `X(-1)` -- so nothing special is needed: the estimator
takes the 2005 value from the data and the regressor `X/X(2005)` is the series divided by that
value.

Some synthetic data where the answer is known: `Y` depends on `X` relative to its 2005 value.

In [12]:
rng = np.random.default_rng(1)
eyears = list(range(2000, 2021))
n = len(eyears)
dfe = pd.DataFrame({'X': np.linspace(10, 30, n) + rng.normal(0, 1, n),
                    'Z': rng.normal(0, 1, n)}, index=eyears)
dfe['Y'] = 2.0 + 3.0 * dfe.X / dfe.loc[2005, 'X'] + 0.5 * dfe.Z + rng.normal(0, 0.05, n)

Estimated with the fixed period, and with the old workaround -- a column holding the 2005 value
in every row. The estimates are the same, and close to the 2, 3 and 0.5 the data was made from:

In [13]:
ols = Estimate_ols(org_eq='Y = C(1) + C(2)*(X/X(2005)) + C(3)*Z',
                   input_df=dfe, smpl=(2001, 2020), ecm=False)
workaround = Estimate_ols(org_eq='Y = C(1) + C(2)*(X/X_AT_2005) + C(3)*Z',
                          input_df=dfe.assign(X_AT_2005=dfe.loc[2005, 'X']),
                          smpl=(2001, 2020), ecm=False)
pd.DataFrame({'X(2005)': ols.coef_estimate_dict, 'workaround': workaround.coef_estimate_dict})

,X(2005),workaround
C__1,2.012592,2.012592
C__2,2.987269,2.987269
C__3,0.499745,0.499745


The non-linear estimator works the same way. It defaults to an error correction equation
(`ecm=True`), which this is not:

In [14]:
nls = Estimate_nls_lmfit(org_eq='Y = C(1) + C(2)*(X/X(2005)) + C(3)*Z',
                         input_df=dfe, smpl=(2001, 2020), ecm=False)
pd.Series(nls.coef_estimate_dict)

C__1    2.012592
C__2    2.987269
C__3    0.499745
dtype: float64

### While building a model

In `Makemodel` (and in `%%Makemymodel`) an equation tagged `<est=...>` is estimated while the
model is built, and `X(2005)` stays a fixed period through estimation and normalization. Here the
estimated equation for `y` is followed by an identity which indexes `y` to 2005:

In [15]:
Mfix = Makemodel("""
## Y relative to the level of X in 2005
><est=ols,smpl=2001 2020> y = c(1) + c(2)*(x/x(2005)) + c(3)*z

## Y indexed to 2005
> w = y/y(2005)*100
""", input_df=dfe)
Mfix.show

FRML <EST=OLS,SMPL=2001 2020> Y = (2.0125924040)+(2.9872690312)*(X/X(2005))+(0.4997447014)*Z $
FRML <> W = Y/Y(2005)*100 $


`w` reads the **endogenous** `y` in 2005. Solved over the whole estimation sample, 2005 is inside
the solve span, so this needs a stacked solver; solved from 2006 on, `y(2005)` is data and any
solver will do:

In [16]:
mfixest = Mfix.mmodel
whole = mfixest(dfe, 2001, 2020, solver='newtonstack_ng')
print('w in 2005:', whole.loc[2005, 'W'])
whole.loc[2003:2008, ['X', 'Z', 'Y', 'W']]

w in 2005: 100.0


,X,Z,Y,W
2003,11.696843,-2.711162,2.919827,59.436448
2004,14.905356,-1.889013,3.951206,80.431350
2005,15.446375,-0.174772,4.912520,100.000000
2006,15.463047,-0.422190,4.792098,97.548679
2007,17.581118,0.213643,5.519480,112.355360
2008,18.364572,0.217322,5.672835,115.477089


## Quick reference

```text
> index = x/x(2015)*100           # a year: unsigned, at least 4 digits
> index = x/x('2015Q1')*100       # a quarter or a month: the index label in quotes
```

* Only the ng solvers evaluate a fixed period; calling the model picks one.
* An endogenous variable at a fixed period **inside** the solve span needs a stacked solver:
  `mymodel(df, start, end, solver='newtonstack_ng')` (or `newtonstack_fbmin_ng`).
* The period must be a row of the DataFrame.
* The estimators and `Makemodel` keep `X(2005)` a fixed period; the estimates equal those of the
  workaround with an exogenous column holding the 2005 value.